# Cartoonize The Image (OpenCV)

This notebook refactors **Cartoonize The Image** into a real notebook-first computer vision project using classical OpenCV stylization.

## Task Family
Classical CV stylization

## Preferred Stack
OpenCV

## Dataset Source
Kaggle Flickr8k: https://www.kaggle.com/datasets/adityajn105/flickr8k

## What This Notebook Does
1. Downloads Flickr8k in notebook code
2. Verifies files and split metadata availability
3. Applies a real cartoonization pipeline
4. Runs honest qualitative and proxy quantitative analysis
5. Saves real outputs and metrics

## 1) Environment Setup

In [ ]:
from pathlib import Path
import json
import random
import subprocess
import sys

import numpy as np
import cv2
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd().resolve().parent
OUTPUT_DIR = PROJECT_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project directory: {PROJECT_DIR}')
print(f'Output directory : {OUTPUT_DIR}')
print(f'OpenCV version   : {cv2.__version__}')

In [ ]:
try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kagglehub'])
    import kagglehub

print('kagglehub is ready')

## 2) Dataset Download and Verification

In [ ]:
import kagglehub

print('Downloading dataset from Kaggle...')
dataset_root = Path(kagglehub.dataset_download('adityajn105/flickr8k')).resolve()
print(f'Dataset root: {dataset_root}')

image_paths = sorted(list(dataset_root.rglob('*.jpg')) + list(dataset_root.rglob('*.jpeg')) + list(dataset_root.rglob('*.png')))
print(f'Total image files found: {len(image_paths)}')

if len(image_paths) == 0:
    raise FileNotFoundError('No images found after download. Cannot continue.')

In [ ]:
txt_files = sorted(dataset_root.rglob('*.txt'))
split_like_files = [p for p in txt_files if any(k in p.name.lower() for k in ['train', 'val', 'test', 'split'])]
caption_like_files = [p for p in txt_files if 'caption' in p.name.lower()]

print('Verification checks:')
print(f'- Dataset root exists: {dataset_root.exists()}')
print(f'- Image count > 0    : {len(image_paths) > 0} ({len(image_paths)})')
print(f'- Text metadata files: {len(txt_files)}')
print(f'- Split-like files   : {len(split_like_files)}')
print(f'- Caption-like files : {len(caption_like_files)}')

if split_like_files:
    print('Sample split file names:')
    for p in split_like_files[:5]:
        print(f'  - {p.name}')
else:
    print('No explicit train/val/test split files were found. For this classical stylization task, this is acceptable because no model training is required.')

In [ ]:
preview_paths = image_paths[:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for i, p in enumerate(preview_paths):
    bgr = cv2.imread(str(p))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    axes[i].imshow(rgb)
    axes[i].set_title(p.name[:32], fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
preview_file = OUTPUT_DIR / 'dataset_preview.png'
plt.savefig(preview_file, dpi=140, bbox_inches='tight')
plt.show()
print(f'Saved: {preview_file}')

## 3) Cartoonization Method (OpenCV)

We use a classical pipeline:
1. Bilateral filtering to smooth color regions while preserving edges
2. Grayscale + median blur + adaptive threshold for edge mask
3. Bitwise combination of smoothed image and edge mask

In [ ]:
def cartoonize_image(bgr: np.ndarray) -> np.ndarray:
    if bgr is None or bgr.size == 0:
        raise ValueError('Input image is empty')

    # Smooth colors but preserve boundaries
    color = cv2.bilateralFilter(bgr, d=9, sigmaColor=200, sigmaSpace=200)

    # Extract bold edges
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.medianBlur(gray, 7)
    edges = cv2.adaptiveThreshold(
        blur,
        maxValue=255,
        adaptiveMethod=cv2.ADAPTIVE_THRESH_MEAN_C,
        thresholdType=cv2.THRESH_BINARY,
        blockSize=9,
        C=2
    )

    # Combine flat colors with black edges
    cartoon = cv2.bitwise_and(color, color, mask=edges)
    return cartoon

## 4) Run Stylization and Save Outputs

In [ ]:
max_images = 24
selected_paths = image_paths[:max_images]

records = []
stylized_dir = OUTPUT_DIR / 'stylized_images'
stylized_dir.mkdir(parents=True, exist_ok=True)

for p in selected_paths:
    original_bgr = cv2.imread(str(p))
    if original_bgr is None:
        continue

    cartoon_bgr = cartoonize_image(original_bgr)
    out_path = stylized_dir / p.name
    cv2.imwrite(str(out_path), cartoon_bgr)

    records.append({
        'image_name': p.name,
        'input_path': str(p),
        'output_path': str(out_path),
        'height': int(original_bgr.shape[0]),
        'width': int(original_bgr.shape[1])
    })

print(f'Processed images: {len(records)}')
print(f'Saved stylized outputs to: {stylized_dir}')

if len(records) == 0:
    raise RuntimeError('No images were processed.')

## 5) Honest Qualitative and Proxy Quantitative Analysis

Because this is non-learning stylization, there is no classification accuracy. We report:
- Edge retention ratio (cartoon edges compared with original edges)
- Color simplification ratio (fewer unique colors after stylization on downscaled images)

In [ ]:
import pandas as pd

def edge_density(gray: np.ndarray) -> float:
    edge = cv2.Canny(gray, 100, 200)
    return float(np.count_nonzero(edge) / edge.size)

def unique_color_count(bgr: np.ndarray, side: int = 128) -> int:
    small = cv2.resize(bgr, (side, side), interpolation=cv2.INTER_AREA)
    flat = small.reshape(-1, 3)
    return int(np.unique(flat, axis=0).shape[0])

analysis_rows = []
for r in records:
    src = cv2.imread(r['input_path'])
    dst = cv2.imread(r['output_path'])

    src_gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
    dst_gray = cv2.cvtColor(dst, cv2.COLOR_BGR2GRAY)

    src_edge_density = edge_density(src_gray)
    dst_edge_density = edge_density(dst_gray)

    src_colors = unique_color_count(src)
    dst_colors = unique_color_count(dst)

    analysis_rows.append({
        'image_name': r['image_name'],
        'src_edge_density': src_edge_density,
        'dst_edge_density': dst_edge_density,
        'edge_retention_ratio': (dst_edge_density / src_edge_density) if src_edge_density > 0 else np.nan,
        'src_unique_colors': src_colors,
        'dst_unique_colors': dst_colors,
        'color_simplification_ratio': (dst_colors / src_colors) if src_colors > 0 else np.nan
    })

analysis_df = pd.DataFrame(analysis_rows)
analysis_csv = OUTPUT_DIR / 'cartoonization_analysis.csv'
analysis_df.to_csv(analysis_csv, index=False)

summary_metrics = {
    'images_analyzed': int(len(analysis_df)),
    'mean_edge_retention_ratio': float(analysis_df['edge_retention_ratio'].mean()),
    'mean_color_simplification_ratio': float(analysis_df['color_simplification_ratio'].mean()),
    'median_edge_retention_ratio': float(analysis_df['edge_retention_ratio'].median()),
    'median_color_simplification_ratio': float(analysis_df['color_simplification_ratio'].median())
}

metrics_json = OUTPUT_DIR / 'metrics.json'
with open(metrics_json, 'w', encoding='utf-8') as f:
    json.dump(summary_metrics, f, indent=2)

print('Summary metrics:')
for k, v in summary_metrics.items():
    print(f'  {k}: {v}')
print(f'Saved: {analysis_csv}')
print(f'Saved: {metrics_json}')

In [ ]:
show_n = min(6, len(records))
fig, axes = plt.subplots(show_n, 2, figsize=(10, 4 * show_n))
if show_n == 1:
    axes = np.array([axes])

for i in range(show_n):
    src = cv2.cvtColor(cv2.imread(records[i]['input_path']), cv2.COLOR_BGR2RGB)
    dst = cv2.cvtColor(cv2.imread(records[i]['output_path']), cv2.COLOR_BGR2RGB)

    axes[i, 0].imshow(src)
    axes[i, 0].set_title(f'Original: {records[i]["image_name"]}', fontsize=9)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(dst)
    axes[i, 1].set_title('Cartoonized', fontsize=9)
    axes[i, 1].axis('off')

plt.tight_layout()
comparison_file = OUTPUT_DIR / 'cartoonization_comparison.png'
plt.savefig(comparison_file, dpi=140, bbox_inches='tight')
plt.show()
print(f'Saved: {comparison_file}')

## 6) Export Manifest and Final Notes

In [ ]:
manifest = {
    'project': 'Cartoonize The Image',
    'task_family': 'classical_cv_stylization',
    'dataset': {
        'source': 'https://www.kaggle.com/datasets/adityajn105/flickr8k',
        'local_root': str(dataset_root),
        'image_count': len(image_paths),
        'split_files_found': len(split_like_files),
        'caption_files_found': len(caption_like_files)
    },
    'processing': {
        'images_processed': len(records),
        'method': 'bilateral_filter + adaptive_threshold + bitwise_and'
    },
    'outputs': {
        'preview': str(preview_file),
        'comparison': str(comparison_file),
        'metrics_json': str(metrics_json),
        'analysis_csv': str(analysis_csv),
        'stylized_dir': str(stylized_dir)
    },
    'notes': [
        'This is a non-learning stylization pipeline, so no classification accuracy is reported.',
        'Proxy quantitative metrics are included for transparency.'
    ]
}

manifest_file = OUTPUT_DIR / 'manifest.json'
with open(manifest_file, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print(f'Saved: {manifest_file}')
print('Notebook run complete.')